In [2]:
import math
import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
import os
import pandas as pd
import json
import pickle
import seaborn as sns
import yaml
from daart.data import DataGenerator, compute_sequence_pad
from daart.eval import get_precision_recall, run_lengths
from daart.io import get_expt_dir
from daart.transforms import ZScore

from daart_utils.data import DataHandler
from daart_utils.models import compute_model_predictions, get_default_hparams
from daart_utils.plotting import plot_heatmaps
import ssm
from ssm.util import random_rotation, find_permutation
import math
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score
import torch
from daart.io import get_expt_dir, find_experiment



def read_file(path):
    with open(path, 'r') as f:
        content = f.read()
        f.close()
    return content

def get_framewise_f1(pred, true, n):
    
    res = []

    for i in range(1, n+1):
        pos_temp = np.zeros_like(true)
        pos_temp[np.where(true==i)] = 1
        
        pred_temp = np.zeros_like(pred)
        pred_temp[np.where(pred==i)] = 1
        
        metrics = {}
        tp = 0
        fp = 0
        fn = 0
        tn = 0
        
        for t, p in zip(pos_temp, pred_temp):
            if t == 1 and p == 1:
                tp+=1
            elif t == 0 and p == 1:
                fp+=1
            elif t == 1 and p ==0:
                fn+=1
            else:
                tn+=1
        
        metrics['tp'] = tp
        metrics['fp'] = fp
        metrics['fn'] = fn
        metrics['n'] = tp+fn
        
        res.append(metrics)
        
    return res
    

In [3]:
sess_ids_test = [
    # test
    '2019_06_26_fly2',
    '2019_08_14_fly1',
    '2019_08_20_fly3',
    '2019_10_14_fly2',
    '2019_10_21_fly1',  
]

parts = ['avg', 'still', 'walk', 'front_groom', 'back_groom', 'abdomen-move']

data_path = '/home/bsb2144/daart_utils/data/'
dataset = 'fly-5'
input_type = 'markers'

sequence_length = 500
batch_size = 2
sequence_pad = 24
model_class='segmenter'
backbone='tcn'
save_path = '/home/bsb2144/daart/metrics/results.json'
model_dir = "/home/bsb2144/daart/results_daart/fly-5/multi-0/dtcn/time_tcn-5-good_sample-0_markers/version_0"


In [4]:
from daart.models import Segmenter
hparams = get_default_hparams(
    model_class=model_class, device='cuda', sequence_length=sequence_length, n_lags=4,
    input_type=input_type, backbone=backbone, batch_size=batch_size,
    anneal_start=25, anneal_end=75, prob_threshold=0.9,  # pseudo_labels params
)
hparams['sequence_pad'] = compute_sequence_pad(hparams)
model_file = os.path.join(model_dir, 'last_model.pt')
arch_file = os.path.join(model_dir, 'hparams.yaml')
with open(arch_file, 'rb') as f:
    hparams_new = yaml.safe_load(f)

model_0 = Segmenter(hparams_new)
model_0.load_state_dict(torch.load(
    model_file, map_location=lambda storage, loc: storage))

model_0.to('cuda')
model_0.eval()


# test data
states_hand = {}
states_0 = {}
n_vids = []
frame_f1 = []
nums = []

meta = {}

for expt_id in sess_ids_test:
    print(expt_id)

    # initialize data handler; point to correct base path
    handler = DataHandler(expt_id, base_path=os.path.join(data_path, dataset))
    if input_type == 'markers':
        markers_file = handler.get_marker_filepath()
    else:
        markers_file = handler.get_feature_filepath(dirname=input_type)

    hand_labels_file = os.path.join(
                "/home/bsb2144/daart/data/", 'labels-hand', expt_id + '_labels.csv')

    # define data generator signals
    signals = ['markers', 'labels_strong']
    transforms = [ZScore(), None]
    paths = [markers_file, hand_labels_file]

    # build data generator
    data_gen_test = DataGenerator(
        [expt_id], [signals], [transforms], [paths], device='cuda',#hparams['device'], 
        batch_size=hparams_new['batch_size'], trial_splits='1;1;0;0', 
        sequence_pad=hparams_new['sequence_pad'], sequence_length=hparams_new['sequence_length'],
        input_type=hparams_new['input_type'])

    # load hand labels
    handler.load_hand_labels()
    states = np.argmax(handler.hand_labels.vals, axis=1)
    states_hand[expt_id] = states

    # compute predictions
    print('computing predictions for model 0...', end='')
    tmp = model_0.predict_labels(data_gen_test, return_scores=True)

    labels_pred = np.vstack(tmp['labels'][0])
    yhat = np.vstack(tmp['labels'][0])
    labels_model = np.argmax(labels_pred, axis=1)
    states_0[expt_id] = labels_model

    states = states[:len(labels_model)]
    states_hand[expt_id] = states

    temp = get_framewise_f1(states_0[expt_id][states > 0], states[states > 0], len(parts)-1)
    frame_f1.append(temp)
    n_vids.append(len(states[states > 0]))

# Framewise F1
res = []
ns = []

for i in range(len(parts)-1):
    metrics = [frame_f1[j][i] for j in range(len(frame_f1))]
    tp = sum([m['tp'] for m in metrics])
    fp = sum([m['fp'] for m in metrics])
    fn = sum([m['fn'] for m in metrics])
    n = sum([m['n'] for m in metrics])
    precision = tp / float(tp + fp+ 1e-6)
    recall = tp / float(tp + fn+ 1e-6)
    f1 = 2.0 * (precision * recall) / (precision + recall + 1e-6)
    res.append(f1)
    ns.append(n)

rolling_f1 = round(np.mean(res),3)
meta['f1_fw'] = list(np.round([rolling_f1] + res, 3))
print('f1 framewise: ', rolling_f1)
print('F1 scores')
for part, score in zip(parts, meta['f1_fw']):
    print("{}: {}".format(part,score))


# with open(save_path, 'w') as json_file:
#     json.dump(meta, json_file)
#     print('saved to {}'.format(json_file))


2019_06_26_fly2
NZ:  0
computing predictions for model 0...

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


2019_08_14_fly1
NZ:  0
computing predictions for model 0...2019_08_20_fly3
NZ:  0
computing predictions for model 0...2019_10_14_fly2
NZ:  0
computing predictions for model 0...2019_10_21_fly1
NZ:  0
computing predictions for model 0...f1 framewise:  0.85
F1 scores
avg: 0.85
still: 0.734
walk: 0.791
front_groom: 0.936
back_groom: 0.855
abdomen-move: 0.934
